In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data/raw/banking77"
cache_dir = project_root / "data/processed/lab10"

raw_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

source_url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)
raw_path = raw_dir / "train.csv"

if not raw_path.exists():
    urlretrieve(source_url, raw_path)

questions = pd.read_csv(raw_path)
questions.insert(0, "question_id", range(len(questions)))

assert questions[["text", "category"]].notna().all().all()

print(f"Questions: {len(questions):,}")
print(f"Categories: {questions['category'].nunique()}")
display(questions.head())

Questions: 10,003
Categories: 77


,question_id,text,category
0,0,I am still waiting on my card?,card_arrival
1,1,What can I do if my card still hasn't arrived ...,card_arrival
2,2,I have been waiting over a week. Is the card s...,card_arrival
3,3,Can I track my card while it is in the process...,card_arrival
4,4,"How do I know if I will get my card, or if it ...",card_arrival


In [2]:
selected_categories = [
    "cancel_transfer",
    "pending_transfer",
    "failed_transfer",
    "declined_transfer",
    "transfer_not_received_by_recipient",
    "balance_not_updated_after_bank_transfer",
    "card_payment_not_recognised",
    "pending_card_payment",
    "transaction_charged_twice",
    "request_refund",
]

assert set(selected_categories).issubset(
    set(questions["category"])
)

selected = questions.loc[
    questions["category"].isin(selected_categories)
].copy()

# A comparison key for finding repeated text.
# Preserve the original text for the model and for inspection.
selected["text_key"] = (
    selected["text"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

assert selected["text_key"].ne("").all()

# Exclude identical normalized texts with conflicting labels.
conflicting = (
    selected.groupby("text_key")["category"]
    .transform("nunique")
    .gt(1)
)

before = len(selected)

selected = (
    selected.loc[~conflicting]
    .drop_duplicates("text_key")
    .reset_index(drop=True)
)

print(f"Selected questions: {len(selected):,}")
print(f"Rows excluded during duplicate checks: {before - len(selected)}")

display(selected["category"].value_counts().sort_index())

Selected questions: 1,588
Rows excluded during duplicate checks: 0


category
balance_not_updated_after_bank_transfer    171
cancel_transfer                            157
card_payment_not_recognised                168
declined_transfer                          133
failed_transfer                            137
pending_card_payment                       159
pending_transfer                           148
request_refund                             169
transaction_charged_twice                  175
transfer_not_received_by_recipient         171
Name: count, dtype: int64

In [6]:
''' 
Why keep them separate? If the query itself were also in the corpus, 
the search could simply find that identical question. 
We want to test whether it can find other questions with a similar meaning.
'''
corpus, queries = train_test_split(
    selected,
    test_size=0.20,
    stratify=selected["category"],
    random_state=42,
)

corpus = corpus.reset_index(drop=True)
queries = queries.reset_index(drop=True)

assert set(corpus["question_id"]).isdisjoint(
    set(queries["question_id"])
)
assert set(corpus["text_key"]).isdisjoint(
    set(queries["text_key"])
)
assert set(corpus["category"]) == set(queries["category"])

split_summary = pd.DataFrame({
    "corpus": corpus["category"].value_counts(),
    "queries": queries["category"].value_counts(),
}).sort_index()

display(split_summary)

display(
    queries[["question_id", "text", "category"]]
    .sample(5, random_state=42)
)

,corpus,queries
category,,
balance_not_updated_after_bank_transfer,137,34
cancel_transfer,126,31
card_payment_not_recognised,134,34
declined_transfer,106,27
failed_transfer,110,27
pending_card_payment,127,32
pending_transfer,118,30
request_refund,135,34
transaction_charged_twice,140,35


,question_id,text,category
73,8701,How long do transfers from a UK bank account u...,balance_not_updated_after_bank_transfer
277,6340,I have been charged twice this week for a coup...,transaction_charged_twice
25,7485,Why can't I transfer my money into someone els...,failed_transfer
256,5548,why was a transfer declined,declined_transfer
9,5466,It has been forever ago that I made an order o...,request_refund


In [7]:
corpus.to_parquet(
    cache_dir / "corpus.parquet",
    index=False,
)
queries.to_parquet(
    cache_dir / "queries.parquet",
    index=False,
)

print(f"Searchable questions: {len(corpus):,}")
print(f"Evaluation queries: {len(queries):,}")

Searchable questions: 1,270
Evaluation queries: 318
